# Universal Text-to-SQL Agent — End-to-End Walkthrough

This notebook is a guided tour of the whole codebase, written for someone
who has never seen it before and wants to understand **how it actually
works**, not just that it works. It runs every layer of the system for
real, in order, printing what each layer produces before handing off to the
next one.

By the end you will have:

- Bootstrapped the agent against a database with zero configuration.
- Inspected the auto-discovered schema, the auto-built knowledge graph, the
  auto-generated business glossary, the semantic (TF-IDF) table ranking, and
  the value/entity grounding index — the five things that let this agent
  work on a database it has never seen before.
- Called every LangGraph node **by hand**, one at a time, watching the
  shared state grow after each call.
- Run the full compiled agent graph end-to-end, including self-consistency
  voting, complexity-aware query decomposition, SQL safety guardrails,
  multi-turn conversational follow-ups, and pluggable LLM providers.
- Run the evaluation harness that regression-tests generation quality.
- Pointed the entire pipeline at a brand-new, unrelated database schema with
  no extra code, as proof the "universal" claim is real.

See [`README.md`](../README.md) for the prose/diagram version of this same
architecture — this notebook is the "watch it happen" companion to that
document's "read about it" version.

## 1. Setup

### Why this notebook runs without an API key

Every test in `tests/` runs with **no LLM API key and no network access** —
they replace the real chat model with a small stand-in built from
[`langchain_core.runnables.RunnableLambda`](https://python.langchain.com/) that
returns pre-written text instead of calling out to a real model. See
`tests/test_self_consistency.py`, `tests/test_decomposition.py`, and
`eval/runner.py::mock_generator` for the pattern this notebook reuses.

This notebook follows the same convention **by default**, for a very
practical reason: it means this notebook actually runs, deterministically,
for *every* reader — no signup, no billing, no flaky network call, no
leaked key in a shared notebook. A single `USE_REAL_LLM` flag below controls
whether the mock is used or a real provider is called instead; flip it to
`True` (and configure a provider — see §1.3) to see live model output
instead of the scripted responses. **Every section in this notebook
actually goes live when you do** — not just a handful of them — since
`llm_for(...)`/`llm_for_diverse(...)` is used consistently throughout
rather than just in a few spots.

Everywhere you see a "the LLM said X" comment below, imagine it's standing
in for what an actual model call would return for that prompt — the
scripted text is chosen to be exactly what a competent model would say, so
the rest of the pipeline behaves identically either way.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

# Make the repo root importable regardless of where this notebook was
# launched from. Jupyter typically sets the kernel's working directory to
# wherever the .ipynb file lives (here, notebooks/), but this repo's `eval`
# package is a plain top-level directory (not pip-installed), so it's only
# importable once the repo root is on sys.path -- universal_text2sql itself
# doesn't need this (it's `pip install -e .`-ed and therefore already
# importable from anywhere), but `eval` (used in Section 15) does.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "eval").is_dir() and (_candidate / "universal_text2sql").is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# The library writes a few small cache/state files to disk by default
# (query memory, the auto-generated glossary cache, the value-grounding
# cache) so that repeated runs against the same database are fast. Point
# all of them at a throwaway temp directory so running this notebook never
# leaves generated files behind in the git repository.
_work_dir = tempfile.mkdtemp(prefix="text2sql_notebook_")
os.environ.setdefault("QUERY_MEMORY_PATH", os.path.join(_work_dir, "query_memory.json"))
os.environ.setdefault("METADATA_CACHE_DIR", os.path.join(_work_dir, "metadata_cache"))
os.environ.setdefault("GROUNDING_CACHE_DIR", os.path.join(_work_dir, "grounding_cache"))

# --- The one flag that matters -------------------------------------------
# False (default): every LLM call in this notebook is a scripted stand-in,
#                   so the notebook runs anywhere, deterministically, with
#                   no API key and no network access -- the same guarantee
#                   the test suite has.
# True:             every LLM call goes to a real provider via
#                   universal_text2sql.llm.factory.get_llm() -- configure
#                   GROQ_API_KEY (or LLM_PROVIDER + that provider's key)
#                   first, e.g. by copying .env.example to .env. Every
#                   section in this notebook actually goes live, not just
#                   a subset -- see Section 10 for one live-only wrinkle
#                   (self-consistency needs a warmer temperature).
USE_REAL_LLM = False

print(f"Scratch directory for this run: {_work_dir}")
print(f"USE_REAL_LLM = {USE_REAL_LLM}")
if USE_REAL_LLM and not os.getenv("GROQ_API_KEY") and os.getenv("LLM_PROVIDER", "groq").lower() == "groq":
    print("\nWARNING: USE_REAL_LLM is True but GROQ_API_KEY is not set.")
    print("Either set it (cp .env.example .env, or export it directly), or set")
    print("USE_REAL_LLM = False above to run this notebook in scripted/mock mode instead.")

Scratch directory for this run: /tmp/text2sql_notebook_6g8jclpw
USE_REAL_LLM = False


### 1.2 Imports

Everything below is imported from the `universal_text2sql` package exactly
as an application built on top of this library would import it — nothing
in this notebook reaches into private (`_`-prefixed) internals except where
explicitly noted for illustration.

In [2]:
import json
import re

import pandas as pd
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

from universal_text2sql.agent import nodes as agent_nodes
from universal_text2sql.agent.graph import run_query
from universal_text2sql.agent.memory import QueryMemory
from universal_text2sql.bootstrap import bootstrap
from universal_text2sql.database.connector import DatabaseConnector
from universal_text2sql.database.safety import (
    SQLSafetyError,
    classify_statement,
    enforce_read_only,
)
from universal_text2sql.database.schema import SchemaDiscovery
from universal_text2sql.knowledge.graph import SchemaKnowledgeGraph
from universal_text2sql.knowledge.grounding import ValueGroundingIndex
from universal_text2sql.knowledge.metadata import MetadataEnricher
from universal_text2sql.llm.factory import get_llm
from universal_text2sql.utils.demo_data import seed_demo_database

pd.set_option("display.max_colwidth", 80)
print("Imports OK.")

Imports OK.


### 1.3 A stand-in LLM, built the same way the test suite builds one

`generate_sql`, `classify_complexity`, and every other node that needs an
LLM only requires an object with a LangChain-compatible `.invoke()` /
`__or__()` — see `universal_text2sql/llm/base.py`'s `LLMRunnable` protocol.
`RunnableLambda` satisfies that protocol trivially by wrapping a plain
Python function, which is exactly what makes it usable as a drop-in stand-in
in both the test suite and here.

Two small helpers are defined below and reused for the rest of the
notebook:

- **`sequence_llm(*responses)`** — returns each string in `responses`, in
  order, one per `.invoke()` call, repeating the last one if more calls
  happen than responses were given. Used wherever the exact number and
  order of LLM calls a section will make is known up front (which is most
  sections — one call per node, in a fixed pipeline order).
- **`llm_for(*responses)`** — returns a real provider-backed LLM
  (`get_llm()`) when `USE_REAL_LLM` is `True`, otherwise
  `sequence_llm(*responses)`. This is the function actually used throughout
  the notebook, so flipping the one flag above is enough to switch every
  example from scripted to live.
- **`llm_for_diverse(*responses, temperature=0.7)`** — the same idea, but
  requests a warmer real model instead of `get_llm()`'s default
  `temperature=0.0`. Used only in Section 10 (self-consistency), which
  specifically needs the model to produce *different* candidates from
  repeated calls on the same prompt — see that section for why.

In [3]:
def sequence_llm(*responses: str) -> RunnableLambda:
    """A stand-in chat model returning `responses` in order, one per call."""
    responses_list = list(responses)
    calls = {"n": 0}

    def _invoke(prompt_value):
        i = min(calls["n"], len(responses_list) - 1)
        calls["n"] += 1
        return AIMessage(content=responses_list[i])

    return RunnableLambda(_invoke)


def llm_for(*responses: str) -> RunnableLambda:
    if USE_REAL_LLM:
        return get_llm()
    return sequence_llm(*responses)


def llm_for_diverse(*responses: str, temperature: float = 0.7):
    """Like `llm_for`, but requests a warmer real model -- see Section 10:
    self-consistency needs temperature > 0 to produce genuinely different
    candidates; get_llm()'s default (0.0) is deliberately deterministic."""
    if USE_REAL_LLM:
        return get_llm(temperature=temperature)
    return sequence_llm(*responses)


print("Mock LLM helpers ready.")

Mock LLM helpers ready.


## 2. The problem this project solves, in one paragraph

Given a natural-language question and *some* database the agent has never
seen configured for it before, produce correct SQL and a correct answer —
with no human writing a schema description, no human curating example
queries, and no human telling it which columns are safe to filter on with
which literal values. That's harder than it sounds for three separate
reasons this codebase addresses with three separate techniques, each with
roots in recent text-to-SQL research (see the module docstrings for the
specific references — `knowledge/graph.py`, `knowledge/metadata.py`,
`knowledge/grounding.py`, `agent/nodes.py`):

1. **Schema linking** — which of possibly dozens of tables/columns are
   relevant to *this* question, and how do you join them if they're not
   directly related? → the knowledge graph (§4) and semantic retrieval (§6).
2. **Vocabulary mismatch** — the question says "revenue", the column is
   named `total_amount`. → the auto-generated business glossary (§5).
3. **Literal grounding** — the question says "usa", the stored value is
   exactly `"USA"`. An LLM has no way to know that without looking. → value
   grounding (§7).

Everything from §8 onward shows how these three pieces feed into the actual
agent graph that turns a question into SQL, executes it, and answers in
natural language — plus the production-safety and multi-model layers built
on top (§11–§14) and how to measure whether any of this is actually working
(§15).

## 3. Sixty-second quickstart

Before looking at any internals: this is the whole point of the library —
one function call connects to a database (or seeds an in-memory demo one if
none is configured) and prepares everything the agent needs; one method
call answers a question.

In [4]:
quickstart_llm = llm_for(
    "SIMPLE",
    "SELECT COUNT(*) AS customer_count FROM customers",
    "VALID",
    "There are 10 customers in total.",
)

# enable_metadata_enrichment=False here only to keep this cell's scripted
# LLM responses lined up 1:1 with the 4 calls the pipeline itself makes
# (classify -> generate -> validate -> answer). Section 5 turns metadata
# enrichment back on and shows exactly what it adds.
ctx = bootstrap(seed_demo=True, llm=quickstart_llm, enable_metadata_enrichment=False)

result = ctx.ask("How many customers do we have?")

print("Generated SQL:\n ", result["generated_sql"])
print("\nFinal answer:", result["final_answer"])
result["execution_result"]

Generated SQL:
  SELECT COUNT(*) AS customer_count FROM customers

Final answer: There are 10 customers in total.


,customer_count
0,10


That's it — no schema description was written, no example queries were
curated, nothing was configured beyond a database connection. Everything
from here on is "how did that actually happen", peeling back one layer at a
time using a hand-built connector so each piece can be inspected in
isolation before it all gets wired back together in §8 onward.

## 4. Layers 1–2: the universal connector and schema discovery

`DatabaseConnector` wraps a single SQLAlchemy `Engine` and exposes only the
primitives the rest of the system needs — because everything above this
layer only talks through this interface, pointing the agent at PostgreSQL
or MySQL instead of SQLite is a connection-string change, not a code
change. `SchemaDiscovery.discover()` then introspects the live database
*once* (via SQLAlchemy's `inspect()`) into a `DatabaseSchema`: every
table's columns (name, type, nullability, primary key), declared foreign
keys, row count, and a handful of sample values per column. This is the
*raw* structural layer — no semantics yet, that starts in §5.

The rest of this notebook builds on the `connector` and `schema` objects
created in the next cell.

In [5]:
connector = DatabaseConnector("sqlite:///:memory:")  # read_only=True by default -- see SQL_READ_ONLY in .env.example
seed_demo_database(connector)

print("Detected dialect:", connector.db_type)
print("Tables:", connector.get_table_names())

schema = SchemaDiscovery(connector).discover()

for name, meta in schema.tables.items():
    print(f"\n{name}  ({meta.row_count} rows)")
    for col in meta.columns:
        pk = "  [PRIMARY KEY]" if col.primary_key else ""
        samples = f"   e.g. {col.sample_values[:3]}" if col.sample_values else ""
        print(f"  - {col.name}: {col.data_type}{pk}{samples}")
    if meta.foreign_keys:
        for fk in meta.foreign_keys:
            print(f"  FK: {fk['constrained_columns']} -> {fk['referred_table']}.{fk['referred_columns']}")

Detected dialect: SQLite
Tables: ['customers', 'order_items', 'orders', 'products']

customers  (10 rows)
  - customer_id: INTEGER  [PRIMARY KEY]   e.g. [1, 2, 3]
  - first_name: TEXT   e.g. ['Alice', 'Bob', 'Carol']
  - last_name: TEXT   e.g. ['Smith', 'Jones', 'White']
  - email: TEXT   e.g. ['alice@example.com', 'bob@example.com', 'carol@example.com']
  - country: TEXT   e.g. ['USA', 'UK', 'Canada']
  - created_at: DATE   e.g. ['2022-01-15', '2022-03-20', '2022-05-10']

order_items  (20 rows)
  - item_id: INTEGER  [PRIMARY KEY]   e.g. [1, 2, 3]
  - order_id: INTEGER   e.g. [1, 1, 2]
  - product_id: INTEGER   e.g. [1, 2, 3]
  - quantity: INTEGER   e.g. [1, 1, 1]
  - unit_price: REAL   e.g. [1299.99, 29.99, 299.99]
  FK: ['product_id'] -> products.['product_id']
  FK: ['order_id'] -> orders.['order_id']

orders  (15 rows)
  - order_id: INTEGER  [PRIMARY KEY]   e.g. [1, 2, 3]
  - customer_id: INTEGER   e.g. [1, 2, 3]
  - order_date: DATE   e.g. ['2023-01-10', '2023-02-14', '2023-03-05'

In [6]:
# DatabaseSchema.to_ddl() renders the same information back out as CREATE
# TABLE text -- this is what actually goes into the SQL-generation prompt.
print(schema.to_ddl())

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT NOT NULL,
    country TEXT NOT NULL,
    created_at DATE NOT NULL
);

CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    FOREIGN KEY (product_id) REFERENCES products(product_id),
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date DATE NOT NULL,
    status TEXT NOT NULL,
    total_amount REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    stock INTEGER NOT NULL
);



## 5. Layer 3a: the schema knowledge graph

Declared foreign keys only capture *declared* relationships. Real-world
databases — especially ones exported without constraints, like a
CSV-backed SQLite dump — are full of undeclared relationships a human would
infer instantly from naming conventions alone (`orders.customer_id` clearly
means `customers.customer_id`, constraint or not). `SchemaKnowledgeGraph`
(`knowledge/graph.py`) builds a directed graph of tables and columns with
four edge kinds — `has_column`, `foreign_key` (declared), `inferred_fk`
(recovered from naming conventions), and `semantic_sibling` (same column
name reused across tables, e.g. two `email` columns) — and uses it for two
things an LLM struggles with on a schema it's never seen: telling it
exactly which columns to join two tables on, including tables with **no**
direct relationship by routing through an intermediate table, and pruning
the prompt down to only the tables/columns a question actually needs.

In [7]:
kg = SchemaKnowledgeGraph.build(schema)

print(kg.describe(list(schema.tables.keys())))

## Table Relationships (Knowledge Graph)
  customers.customer_id = orders.customer_id  (declared FK)
  order_items.order_id = orders.order_id  (declared FK)
  order_items.product_id = products.product_id  (declared FK)


In [8]:
# order_items and customers have no direct foreign key -- but they're both
# related to orders, so a join path exists by routing through it.
hops = kg.find_join_path("order_items", "customers")

print("Multi-hop join path: order_items -> customers")
for hop in hops:
    print(f"  {hop.as_sql()}   (via a {hop.kind.replace('_', ' ')})")

Multi-hop join path: order_items -> customers
  order_items.order_id = orders.order_id   (via a foreign key)
  orders.customer_id = customers.customer_id   (via a foreign key)


In [9]:
# The graph also renders as a Graphviz DOT string (this is exactly what
# app.py hands to st.graphviz_chart() for the Streamlit sidebar panel).
dot = kg.to_graphviz()
print(dot)

try:
    import graphviz

    display(graphviz.Source(dot))
except Exception as exc:
    print(f"\n(inline rendering skipped: {exc})")
    print("Paste the DOT text above into https://dreampuf.github.io/GraphvizOnline/ to view it visually.")

digraph schema {
  rankdir="LR";
  node [shape=box, fontsize=10];
  "customers" [style=filled, fillcolor="#e8f0fe"];
  "order_items" [style=filled, fillcolor="#e8f0fe"];
  "orders" [style=filled, fillcolor="#e8f0fe"];
  "products" [style=filled, fillcolor="#e8f0fe"];
  "customers" -> "orders" [label="customer_id=customer_id", style=solid, dir=none];
  "order_items" -> "products" [label="product_id=product_id", style=solid, dir=none];
  "order_items" -> "orders" [label="order_id=order_id", style=solid, dir=none];
}

(inline rendering skipped: No module named 'graphviz')
Paste the DOT text above into https://dreampuf.github.io/GraphvizOnline/ to view it visually.


## 6. Layer 3b: the auto-generated business glossary

Raw DDL is a poor proxy for what a column *means* — `t.st` tells an LLM
nothing, "order status, one of pending/shipped/completed/cancelled" does.
`MetadataEnricher` (`knowledge/metadata.py`) generates that semantic layer
itself, in two tiers:

1. **Heuristic pass** (always on, zero cost, no LLM): identifiers are split
   on `snake_case`/`camelCase` boundaries into readable phrases, and role
   is inferred from primary/foreign-key status and sample values.
2. **LLM pass** (optional, one call per table, cached to disk by schema
   signature): asks the model to write a short description of the table
   and each column, plus **synonyms** — alternate terms a user might say
   instead of the literal column name. This is what lets a question about
   "revenue" resolve to a column literally named `total_amount`.

The cell below runs both passes side by side against the same table so the
difference is concrete.

In [10]:
heuristic_enricher = MetadataEnricher(llm=None, enabled=True)
heuristic_enricher.enrich(schema)

print("--- Heuristic-only glossary for `orders` ---\n")
print(heuristic_enricher.glossary_block(schema, ["orders"]))

--- Heuristic-only glossary for `orders` ---

## Business Glossary (auto-generated)
- orders: Table storing orders records.
    - order_id: Unique identifier for each order row (primary key).
    - customer_id: Reference to a customer record (foreign key).
    - order_date: Order date. Example values: 2023-01-10, 2023-02-14, 2023-03-05.
    - status: Status. Example values: completed, completed, shipped.
    - total_amount: Total amount. Example values: 1329.98, 299.99, 529.98.


In [11]:
def metadata_mock(prompt_value):
    """Simulates what an LLM would answer for METADATA_GENERATION_PROMPT.

    Reads which table the prompt is asking about out of the rendered
    prompt text (`Table: <name>`), and, for `orders`, returns a synonym for
    `total_amount` -- exactly the "revenue" -> `total_amount` story the
    business glossary exists to solve. Other tables get a generic
    description with no columns filled in, which MetadataEnricher applies
    without error (it only overwrites fields present in the response).
    """
    text = prompt_value.to_string()
    match = re.search(r"Table:\s*(\w+)", text)
    table = match.group(1) if match else "unknown"

    payload = {"description": f"Stores one row per {table} record.", "synonyms": [], "columns": {}}
    if table == "orders":
        payload["description"] = "Stores one row per customer purchase order."
        payload["synonyms"] = ["sales", "purchases"]
        payload["columns"]["total_amount"] = {
            "meaning": "The total monetary value of the order.",
            "synonyms": ["revenue", "order value", "sales amount"],
        }
        payload["columns"]["status"] = {
            "meaning": "Current fulfillment status of the order.",
            "synonyms": ["order state"],
        }
    return AIMessage(content=json.dumps(payload))


glossary_llm = get_llm() if USE_REAL_LLM else RunnableLambda(metadata_mock)

# A fresh schema copy so the heuristic-only version above stays untouched.
schema_llm_glossary = SchemaDiscovery(connector).discover()
llm_enricher = MetadataEnricher(llm=glossary_llm, enabled=True)
llm_enricher.enrich(schema_llm_glossary)

print("--- Heuristic + LLM-assisted glossary for `orders` ---\n")
print(llm_enricher.glossary_block(schema_llm_glossary, ["orders"]))

--- Heuristic + LLM-assisted glossary for `orders` ---

## Business Glossary (auto-generated)
- orders: Stores one row per customer purchase order. (aka: sales, purchases)
    - order_id: Unique identifier for each order row (primary key).
    - customer_id: Reference to a customer record (foreign key).
    - order_date: Order date. Example values: 2023-01-10, 2023-02-14, 2023-03-05.
    - status: Current fulfillment status of the order. (aka: order state)
    - total_amount: The total monetary value of the order. (aka: revenue, order value, sales amount)


## 7. Layer 3c: semantic (TF-IDF) schema linking

A question rarely uses a table or column's literal name. `retrieval/semantic.py`
is a small, dependency-free TF-IDF + cosine-similarity index (no torch, no
sklearn, no embedding API — the schema/glossary text and questions here are
short enough that classic TF-IDF captures most of the useful signal, stays
deterministic for tests, and needs no extra API dependency). Compare it
against naive literal keyword overlap on the *same*, glossary-enriched
schema from §5 — the word "revenue" is the perfect stress test, since it
appears nowhere as a literal column name.

In [12]:
question = "What is our total revenue?"

naive = schema_llm_glossary.get_relevant_tables(["revenue"])
semantic = schema_llm_glossary.get_relevant_tables_semantic(question)

print("Naive literal keyword-overlap ranking (no column is literally named")
print("'revenue', so this falls back to returning every table, unranked):")
print(" ", naive)

print("\nTF-IDF semantic ranking (finds `orders` via the glossary synonym")
print("'revenue' -> total_amount attached to it in Section 6):")
print(" ", semantic)

Naive literal keyword-overlap ranking (no column is literally named
'revenue', so this falls back to returning every table, unranked):
  ['customers', 'order_items', 'orders', 'products']

TF-IDF semantic ranking (finds `orders` via the glossary synonym
'revenue' -> total_amount attached to it in Section 6):
  ['orders']


## 8. Layer 3d: value/entity grounding

An LLM writing `WHERE country = 'USA'` has to *guess* the exact stored
spelling — `"USA"`? `"United States"`? `"US"`? Neither the schema DDL nor
the glossary tells it. `ValueGroundingIndex` (`knowledge/grounding.py`)
closes that gap in two stages: **profiling** — for every low-cardinality
text column below a row-count ceiling (bounded cost regardless of database
size), probe `COUNT(DISTINCT col)`, and if it's small in both absolute and
relative terms, cache its actual distinct values — and **matching** —
compare 1–3 word n-grams of the question against those cached values
(substring containment + `difflib`, or the optional `rapidfuzz` package for
stronger fuzzy matching) so the SQL-generation prompt can say "the question
says 'usa', the real stored value is `'USA'`" instead of making the LLM
guess.

**Known limitation, stated plainly**: neither matching mode understands a
true synonym with no shared substring, e.g. "USA" vs. "United States" — that
class of gap is instead partly covered by the glossary's LLM-generated
synonyms from §5, not by this module.

In [13]:
grounding_index = ValueGroundingIndex.build(schema, connector)

print(f"Profiled {len(grounding_index.profiles)} categorical column(s):\n")
for p in grounding_index.profiles:
    print(f"  {p.table}.{p.column}: {p.distinct_values}")

Profiled 3 categorical column(s):

  customers.country: ['Australia', 'Canada', 'France', 'Germany', 'Japan', 'UK', 'USA']
  orders.status: ['cancelled', 'completed', 'shipped']
  products.category: ['Appliances', 'Electronics', 'Furniture', 'Stationery']


In [14]:
matches = grounding_index.match("How many customers are from usa?", ["customers"])
print("Matches for the question's terms against real column values:")
print(" ", matches)

print("\nRendered as a prompt block (this is what gets injected into `grounding_hints`):\n")
print(grounding_index.hints_block(matches))

Matches for the question's terms against real column values:
  [('customers', 'country', 'USA')]

Rendered as a prompt block (this is what gets injected into `grounding_hints`):

## Value Grounding Hints (matched against real column data)
  customers.country contains the value: 'USA'


## 9. Layer 4: the agent, one node at a time

Everything above feeds into a LangGraph `StateGraph` — a pipeline of plain
functions (**nodes**), each receiving the current shared state (an
`AgentState` `TypedDict`, `agent/state.py`) and returning a *partial*
update to it (`agent/nodes.py`). This section calls each node **by hand**,
in the order the real graph calls them, printing exactly what state each
one adds — this is the most direct possible answer to "how does this
actually work".

The question chosen is a single-table aggregation, so it takes the
simplest path through the graph: no self-consistency branching (only one
SQL candidate gets generated), no query decomposition, no self-reflection
retry (the SQL succeeds first try), no multi-turn context (no prior
conversation). §11–§14 exercise each of those branches explicitly by
running the *compiled* graph instead of individual nodes.

In [15]:
memory = QueryMemory(enabled=False)  # disabled so this walkthrough's few-shot examples stay empty and predictable

state = {
    "question": "Which country do most of our customers come from?",
    "relevant_tables": [],
    "schema_context": "",
    "column_samples": "",
    "kg_context": "",
    "business_glossary": "",
    "grounding_hints": "",
    "conversation_context": "",
    "complexity": "",
    "sql_candidates": [],
    "sub_questions": [],
    "decomposition_steps": [],
    "few_shot_examples": [],
    "generated_sql": "",
    "execution_result": None,
    "execution_error": "",
    "validation_verdict": "",
    "final_answer": "",
    "retry_count": 0,
    "max_retries": 3,
    "messages": [],
    "success": False,
}


def apply(update: dict, label: str) -> None:
    print(f"--- after {label} ---")
    for key, value in update.items():
        if key == "messages":
            continue
        preview = str(value)
        if len(preview) > 260:
            preview = preview[:260] + " ...(truncated)"
        print(f"  {key} = {preview}")
    state.update(update)
    print()


node_llm = llm_for(
    "MODERATE",
    "SELECT country, COUNT(*) AS customer_count FROM customers GROUP BY country ORDER BY customer_count DESC",
    "VALID",
    "Most customers come from the USA, with 3 customers -- the largest single country in the data.",
)

In [16]:
apply(
    agent_nodes.select_schema(
        state,
        schema=schema,
        memory=memory,
        knowledge_graph=kg,
        metadata_enricher=heuristic_enricher,
        grounding_index=grounding_index,
    ),
    "select_schema",
)

--- after select_schema ---
  relevant_tables = ['customers']
  schema_context = CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT NOT NULL,
    country TEXT NOT NULL,
    created_at DATE NOT NULL
);

  column_samples =   customers.customer_id: [1, 2, 3]
  customers.first_name: [Alice, Bob, Carol]
  customers.last_name: [Smith, Jones, White]
  customers.email: [alice@example.com, bob@example.com, carol@example.com]
  customers.country: [USA, UK, Canada]
  customers.created_at ...(truncated)
  kg_context = ## Table Relationships (Knowledge Graph)
  customers.customer_id = orders.customer_id  (declared FK)
  business_glossary = ## Business Glossary (auto-generated)
- customers: Table storing customers records.
    - customer_id: Unique identifier for each customer row (primary key).
    - first_name: First name. Example values: Alice, Bob, Carol.
    - last_name: Last name. Example v ...(truncated)
  gr

In [17]:
apply(agent_nodes.classify_complexity(state, llm=node_llm), "classify_complexity")

--- after classify_complexity ---
  complexity = MODERATE



In [18]:
apply(agent_nodes.generate_sql(state, llm=node_llm, db_type=connector.db_type), "generate_sql")

--- after generate_sql ---
  generated_sql = SELECT country, COUNT(*) AS customer_count FROM customers GROUP BY country ORDER BY customer_count DESC
  sql_candidates = ['SELECT country, COUNT(*) AS customer_count FROM customers GROUP BY country ORDER BY customer_count DESC']
  execution_error = 



In [19]:
# A no-op here: select_best_candidate only does work (executes every
# candidate and votes) when generate_sql produced more than one candidate,
# which only happens under self-consistency sampling -- see Section 11.
apply(agent_nodes.select_best_candidate(state, connector=connector), "select_best_candidate")

--- after select_best_candidate ---



In [20]:
apply(agent_nodes.execute_sql(state, connector=connector), "execute_sql")

--- after execute_sql ---
  execution_result =      country  customer_count
0        USA               3
1         UK               2
2      Japan               1
3    Germany               1
4     France               1
5     Canada               1
6  Australia               1
  execution_error = 
  success = True



In [21]:
apply(agent_nodes.validate_result(state, llm=node_llm), "validate_result")

--- after validate_result ---
  validation_verdict = VALID
  success = True



In [22]:
apply(agent_nodes.format_answer(state, llm=node_llm), "format_answer")

--- after format_answer ---
  final_answer = Most customers come from the USA, with 3 customers -- the largest single country in the data.
  success = True



In [23]:
apply(agent_nodes.store_memory(state, memory=memory), "store_memory")

print("=" * 60)
print("Final answer:", state["final_answer"])
state["execution_result"]

--- after store_memory ---

Final answer: Most customers come from the USA, with 3 customers -- the largest single country in the data.


,country,customer_count
0,USA,3
1,UK,2
2,Japan,1
3,Germany,1
4,France,1
5,Canada,1
6,Australia,1


## 10. The compiled graph, end-to-end, with self-consistency voting

Calling nodes by hand is instructive but not how the agent is actually
used — `run_query()` (`agent/graph.py`) compiles all of them into a real
LangGraph `StateGraph` with conditional routing and runs it in one call.

This section also turns on **self-consistency** (`self_consistency_samples`,
CHASE-SQL-style): for non-trivial questions, `generate_sql` samples several
SQL candidates instead of one, `select_best_candidate` *executes every one
of them* and keeps whichever SQL text the largest group of candidates agree
with on the actual result — not on matching text. The three scripted
candidates below (used in mock mode) are deliberately chosen to make that
distinction concrete: two are textually different but semantically
identical (same join, different alias style), and the third contains a
genuine mistake an LLM might plausibly make (grouping by the wrong
column) — self-consistency should out-vote the mistake even though
nothing here checks the SQL for correctness directly. In live mode the
actual candidates come from the model itself instead — whatever it
produces across three warmer-temperature samples — and the same
execution-based voting logic decides the winner.

**A live-only wrinkle worth knowing**: `get_llm()`'s default temperature is
`0.0` (`llm/groq_client.py` — deterministic SQL is usually what you want),
but self-consistency's whole premise is sampling *different* candidates
from the *same* prompt. At temperature 0 those repeat calls come back
near-identical, `generate_sql`'s own dedup collapses them to one candidate,
and voting has nothing to do — silently, no error. So when `USE_REAL_LLM`
is `True`, this section specifically asks for a warmer model
(`llm_for_diverse`, temperature `0.7`) instead of the notebook's usual
`llm_for`, purely so the three samples actually have a chance to diverge.
In mock mode the three candidates below are supplied directly either way.

In [24]:
candidate_a = """
SELECT c.first_name || ' ' || c.last_name AS customer,
       COUNT(o.order_id) AS order_count,
       SUM(o.total_amount) AS total_spent
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
""".strip()

candidate_b = """
SELECT customers.first_name || ' ' || customers.last_name AS customer,
       COUNT(orders.order_id) AS order_count,
       SUM(orders.total_amount) AS total_spent
FROM customers
JOIN orders ON orders.customer_id = customers.customer_id
GROUP BY customers.customer_id
ORDER BY total_spent DESC
""".strip()

# A plausible mistake: grouping by country instead of by individual
# customer -- a different (wrong) shape of result.
candidate_c = """
SELECT c.country AS customer,
       COUNT(o.order_id) AS order_count,
       SUM(o.total_amount) AS total_spent
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.country
ORDER BY total_spent DESC
""".strip()

self_consistency_llm = llm_for_diverse(
    "MODERATE",
    candidate_a,
    candidate_b,
    candidate_c,
    "VALID",
    "Here is each customer's order count and total spend, ranked by total spend descending.",
)

result_sc = run_query(
    question="How many orders has each customer placed, and what's their total spend?",
    connector=connector,
    schema=schema,
    memory=QueryMemory(enabled=False),
    llm=self_consistency_llm,
    knowledge_graph=kg,
    metadata_enricher=heuristic_enricher,
    grounding_index=grounding_index,
    self_consistency_samples=3,
)

print(f"Sampled {len(result_sc['sql_candidates'])} candidate(s):")
for i, sql in enumerate(result_sc["sql_candidates"], 1):
    print(f"\n--- candidate {i} ---\n{sql}")

print("\n" + "=" * 60)
print("Winning SQL (whichever text the LARGEST group of candidates agree")
print("on the RESULT of wins -- not whichever text came first):\n")
print(result_sc["generated_sql"])
result_sc["execution_result"]

Sampled 3 candidate(s):

--- candidate 1 ---
SELECT c.first_name || ' ' || c.last_name AS customer,
       COUNT(o.order_id) AS order_count,
       SUM(o.total_amount) AS total_spent
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC

--- candidate 2 ---
SELECT customers.first_name || ' ' || customers.last_name AS customer,
       COUNT(orders.order_id) AS order_count,
       SUM(orders.total_amount) AS total_spent
FROM customers
JOIN orders ON orders.customer_id = customers.customer_id
GROUP BY customers.customer_id
ORDER BY total_spent DESC

--- candidate 3 ---
SELECT c.country AS customer,
       COUNT(o.order_id) AS order_count,
       SUM(o.total_amount) AS total_spent
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.country
ORDER BY total_spent DESC

Winning SQL (whichever text the LARGEST group of candidates agree
on the RESULT of wins -- not whichever text came first):

SELECT c.first_name |

,customer,order_count,total_spent
0,Bob Jones,2,1599.98
1,Alice Smith,2,1409.97
2,Eve Taylor,2,1039.96
3,Carol White,2,829.97
4,Grace Martin,1,499.99
5,Iris Thomas,1,459.98
6,Frank Wilson,1,379.99
7,Jack Jackson,1,159.98
8,David Brown,2,104.97
9,Henry Anderson,1,29.99


## 11. Complexity-aware routing and query decomposition

`classify_complexity` labels every question SIMPLE / MODERATE / COMPLEX
(DIN-SQL-style difficulty routing) before generation ever happens. That
label already gated self-consistency sampling above (SIMPLE questions
always get exactly one candidate); it also gates an entirely different
strategy for COMPLEX questions when `enable_query_decomposition=True`:
`decompose_sql` (`agent/nodes.py`) plans an ordered list of sub-questions
with one LLM call, generates a SQL fragment for each sub-question with one
LLM call apiece, and composes them into a single query via CTEs —
`WITH step_1 AS (...), step_2 AS (...) <final SELECT>` — rather than asking
for one large, intricate query in a single shot. If any step fails to
parse or generate, this falls back to ordinary single-shot generation
rather than ever emitting broken SQL.

`decompose_sql` and self-consistency sampling are mutually exclusive for a
given question in this version — both already multiply LLM calls, and
stacking them would multiply cost further for unclear benefit.

**A live-only wrinkle worth knowing**: whether decomposition fires at all
depends on a real `classify_complexity` call landing on exactly `"COMPLEX"`
(`agent/graph.py::_should_decompose`). The question below — a per-category
"best in class" lookup that genuinely needs a nested/correlated comparison,
not just one `GROUP BY` — is chosen to make that classification likely, but
a model's judgment call is still a judgment call. The code below prints
whichever path actually ran and explains it either way, rather than
assuming decomposition definitely happened.

In [25]:
sub_questions = [
    "Compute each product's total revenue (quantity times unit price, summed "
    "across its order line items), grouped by product and category",
    "For each category, find the maximum total revenue among its products",
    "Identify which specific product achieved that maximum revenue in each "
    "category, and report the category, product name, and revenue",
]

step_1_sql = """
SELECT p.product_id, p.category, p.name,
       SUM(oi.quantity * oi.unit_price) AS product_revenue
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.product_id, p.category, p.name
""".strip()

step_2_sql = """
SELECT category, MAX(product_revenue) AS max_revenue
FROM step_1
GROUP BY category
""".strip()

# The FINAL step returns a complete SELECT (no leading WITH) and may
# reference the earlier steps' aliases as if they were ordinary tables.
step_3_sql = """
SELECT step_1.category, step_1.name AS top_product, step_1.product_revenue
FROM step_1
JOIN step_2 ON step_2.category = step_1.category AND step_2.max_revenue = step_1.product_revenue
ORDER BY step_1.category
""".strip()

decomposition_llm = llm_for(
    "COMPLEX",                        # classify_complexity
    json.dumps(sub_questions),        # decompose_sql's planning call
    step_1_sql,                       # decompose_sql's sub-question 1
    step_2_sql,                       # decompose_sql's sub-question 2
    step_3_sql,                       # decompose_sql's sub-question 3 (final)
    "VALID",                          # validate_result
    "Here is the top-revenue product in each category.",  # format_answer
)

result_decomp = run_query(
    question="For each product category, which single product generated the most total revenue, and how much was that?",
    connector=connector,
    schema=schema,
    memory=QueryMemory(enabled=False),
    llm=decomposition_llm,
    knowledge_graph=kg,
    metadata_enricher=heuristic_enricher,
    grounding_index=grounding_index,
    enable_query_decomposition=True,
)

print(f"Classified complexity: {result_decomp['complexity']}")

if result_decomp["sub_questions"]:
    print(f"\nDecomposition fired -- {len(result_decomp['sub_questions'])} planned sub-question(s):")
    for i, q in enumerate(result_decomp["sub_questions"], 1):
        print(f"  {i}. {q}")
    print("\nComposed SQL (fragments stitched together as CTEs):\n")
else:
    print(
        "\nDecomposition did NOT fire this run -- either the classifier didn't judge"
        "\nthis question COMPLEX enough, or the planning step itself didn't produce"
        "\nusable sub-questions, in which case decompose_sql safely falls back to"
        "\nordinary single-shot generation rather than ever emitting broken SQL."
        "\nBoth outcomes are legitimate; the SQL below is whichever path actually ran:\n"
    )

print(result_decomp["generated_sql"])
result_decomp["execution_result"]

Classified complexity: COMPLEX

Decomposition fired -- 3 planned sub-question(s):
  1. Compute each product's total revenue (quantity times unit price, summed across its order line items), grouped by product and category
  2. For each category, find the maximum total revenue among its products
  3. Identify which specific product achieved that maximum revenue in each category, and report the category, product name, and revenue

Composed SQL (fragments stitched together as CTEs):

WITH step_1 AS (
SELECT p.product_id, p.category, p.name,
       SUM(oi.quantity * oi.unit_price) AS product_revenue
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.product_id, p.category, p.name
),
step_2 AS (
SELECT category, MAX(product_revenue) AS max_revenue
FROM step_1
GROUP BY category
)
SELECT step_1.category, step_1.name AS top_product, step_1.product_revenue
FROM step_1
JOIN step_2 ON step_2.category = step_1.category AND step_2.max_revenue = step_1.product_revenue
ORDE

,category,top_product,product_revenue
0,Appliances,Coffee Maker,89.99
1,Electronics,Laptop Pro,5199.96
2,Furniture,Standing Desk,999.98
3,Stationery,Pen Set,29.97


## 12. SQL safety guardrails

The agent executes LLM-generated SQL against a live database with no human
in the loop — before `database/safety.py` existed, `execute_query` ran
*any* SQL string unconditionally, `SELECT` and `DROP TABLE` treated
identically. `DatabaseConnector` now defaults to `read_only=True`
(`SQL_READ_ONLY` env var): every statement is classified READ or WRITE —
a zero-dependency regex baseline always runs, and an optional `sqlglot`
AST-based check (installed via the `safety` extra) catches the harder case
a regex can't, a `WITH x AS (SELECT ...) DELETE FROM x` CTE that *looks*
read-only from its first keyword but ends in a write — and anything not
confidently classified `READ` is blocked before it ever touches the
database.

In [26]:
print(classify_statement("SELECT * FROM customers"))
print(classify_statement("DELETE FROM customers"))
print(classify_statement("SELECT 1; DROP TABLE customers;"))  # multi-statement -> always WRITE, fails closed

Multi-statement SQL classified as WRITE (unsafe): 2 statements


StatementType.READ
StatementType.WRITE
StatementType.WRITE


In [27]:
try:
    enforce_read_only("DELETE FROM customers")
except SQLSafetyError as exc:
    print("enforce_read_only() blocked it:\n ", exc)

enforce_read_only() blocked it:
  Refusing to execute a non-read-only statement (classified as WRITE). Pass allow_writes=True on execute_query(), or construct DatabaseConnector(read_only=False), to permit this.


In [28]:
# The `connector` built back in Section 4 is read_only=True by default --
# the exact same protection is wired straight into execute_query().
try:
    connector.execute_query("DELETE FROM customers")
except SQLSafetyError as exc:
    print("DatabaseConnector blocked it too:\n ", exc)

DatabaseConnector blocked it too:
  Refusing to execute a non-read-only statement (classified as WRITE). Pass allow_writes=True on execute_query(), or construct DatabaseConnector(read_only=False), to permit this.


In [29]:
# Explicit, deliberate opt-in on a disposable connector -- writes are never
# silently allowed, they require read_only=False at construction time (or
# execute_query(..., enforce_read_only=False) for a single call).
scratch = DatabaseConnector("sqlite:///:memory:", read_only=False)
seed_demo_database(scratch)

before = scratch.execute_query("SELECT COUNT(*) AS n FROM customers")
scratch.execute_query("DELETE FROM customers WHERE customer_id = 10")
after = scratch.execute_query("SELECT COUNT(*) AS n FROM customers")

print(f"read_only=False permitted the DELETE: {before['n'][0]} -> {after['n'][0]} customers")

read_only=False permitted the DELETE: 10 -> 9 customers


## 13. Multi-turn conversational follow-ups

Real usage is a conversation, not a single isolated question — "now filter
that by country" only makes sense with the previous question and answer in
view. This is deliberately **caller-owned** state, not something
`AgentContext` accumulates internally: each call to `ctx.ask(question,
conversation_history=...)` takes the growing list of prior
`{"question", "sql", "answer"}` turns and is responsible for appending its
own result before the next call (see `main.py`'s interactive loop and
`app.py`'s `st.session_state.conversation_history` for the two real
call sites). `run_query()` renders the last few turns into a
"Previous Turn(s)" prompt block (`build_conversation_context_block`,
`prompts/templates.py`) injected into `SQL_GENERATION_PROMPT` alongside a
chain-of-thought step that specifically asks the model to resolve "that" /
"it" against it.

In [30]:
# In mock mode the SQL for turn 2 below is simply pre-written to already
# be the "Canada" version, standing in for what a real model would produce
# after reading the conversation context printed at the end of this cell.
# Flip USE_REAL_LLM = True at the top of the notebook to see a live model
# actually resolve "the same thing... for Canada instead" itself.
conversation_llm = llm_for(
    "MODERATE",
    "SELECT COUNT(*) AS order_count FROM orders o JOIN customers c ON c.customer_id = o.customer_id WHERE c.country = 'USA'",
    "VALID",
    "Customers from the USA placed 4 orders in total.",
    "MODERATE",
    "SELECT COUNT(*) AS order_count FROM orders o JOIN customers c ON c.customer_id = o.customer_id WHERE c.country = 'Canada'",
    "VALID",
    "Customers from Canada placed 2 orders in total.",
)

conversation_ctx = bootstrap(seed_demo=True, llm=conversation_llm, enable_metadata_enrichment=False)
history: list[dict[str, str]] = []

turn_1 = conversation_ctx.ask(
    "How many orders came from customers in the USA?", conversation_history=history
)
print("Turn 1 answer:", turn_1["final_answer"])

# Caller-owned: the context is only carried forward because we append it
# ourselves before the next call.
history.append(
    {
        "question": "How many orders came from customers in the USA?",
        "sql": turn_1["generated_sql"],
        "answer": turn_1["final_answer"],
    }
)

turn_2 = conversation_ctx.ask("Now show me the same thing for Canada instead.", conversation_history=history)
print("Turn 2 answer:", turn_2["final_answer"])

print("\nThe conversation context actually injected into turn 2's prompt:\n")
print(turn_2["conversation_context"])

Turn 1 answer: Customers from the USA placed 4 orders in total.


Turn 2 answer: Customers from Canada placed 2 orders in total.

The conversation context actually injected into turn 2's prompt:

## Previous Turn(s) In This Conversation
Previous question: How many orders came from customers in the USA?
Previous SQL: SELECT COUNT(*) AS order_count FROM orders o JOIN customers c ON c.customer_id = o.customer_id WHERE c.country = 'USA'
Previous answer: Customers from the USA placed 4 orders in total.


## 14. Pluggable LLM providers

Every node above only ever depended on the `LLMRunnable` protocol, never on
a specific vendor SDK — so which model actually answers is a configuration
choice, not a code change. `llm/factory.py::get_llm()` resolves the
`LLM_PROVIDER` env var (default `"groq"`) to one of four lazily-imported
clients; each provider's SDK is an *optional* dependency (`langchain-openai`,
`langchain-anthropic`, `langchain-ollama`) so importing the factory itself
never requires any of them to be installed — only the one actually
selected, and with a clear install hint if it's missing.

In [31]:
print("Resolved provider with LLM_PROVIDER unset:", os.getenv("LLM_PROVIDER", "groq"), "(the default)")

requirements = {
    "groq": "GROQ_API_KEY",
    "openai": "OPENAI_API_KEY   (pip install -e '.[openai]')",
    "anthropic": "ANTHROPIC_API_KEY   (pip install -e '.[anthropic]')",
    "ollama": "a reachable OLLAMA_BASE_URL, no API key needed   (pip install -e '.[ollama]')",
}
for provider, needs in requirements.items():
    print(f"  {provider:10s} needs: {needs}")

try:
    get_llm(provider="not-a-real-provider")
except ValueError as exc:
    print("\nAn unknown provider name raises immediately and clearly:\n ", exc)

print("\nSwitching providers for the rest of an application is one line, e.g.:")
print('  os.environ["LLM_PROVIDER"] = "openai"')
print("  llm = get_llm()   # now returns a ChatOpenAI instance instead of ChatGroq")

Resolved provider with LLM_PROVIDER unset: groq (the default)
  groq       needs: GROQ_API_KEY
  openai     needs: OPENAI_API_KEY   (pip install -e '.[openai]')
  anthropic  needs: ANTHROPIC_API_KEY   (pip install -e '.[anthropic]')
  ollama     needs: a reachable OLLAMA_BASE_URL, no API key needed   (pip install -e '.[ollama]')

An unknown provider name raises immediately and clearly:
  Unknown LLM_PROVIDER 'not-a-real-provider'. Supported: ['anthropic', 'groq', 'ollama', 'openai']

Switching providers for the rest of an application is one line, e.g.:
  os.environ["LLM_PROVIDER"] = "openai"
  llm = get_llm()   # now returns a ChatOpenAI instance instead of ChatGroq


## 15. The evaluation harness — did any of this actually help?

"It answered my question correctly" for one hand-picked question isn't
evidence a prompt or logic change actually helped. `eval/` runs a
golden-question **execution-accuracy** regression suite: each question in
`eval/golden/demo_db.jsonl` has a hand-computed expected result; the
generated SQL is executed and the resulting DataFrame is compared
*order-independently* against that expectation (reusing
`agent/scoring.py::result_signature`, the exact same logic
`select_best_candidate`'s self-consistency voting uses above) — deliberately
**not** an exact-SQL-text match, since two correct queries can differ
arbitrarily in formatting, alias names, or join order.

This cell reproduces exactly what `python -m eval.cli --mock` (also
`make eval`, and wired into CI) reports, using the same zero-API-key
`--mock` mode: a canned per-question SQL lookup instead of a live model.

In [32]:
from eval.runner import load_golden_set, mock_generator, run_golden_set

eval_connector = DatabaseConnector("sqlite:///:memory:", read_only=False)  # mirrors eval/cli.py's _build_demo_connector
seed_demo_database(eval_connector)

dataset = load_golden_set()
summary = run_golden_set(dataset, mock_generator(dataset), eval_connector.execute_query)

for r in summary.results:
    status = "PASS" if r.correct else ("ERROR" if r.execution_error else "FAIL")
    print(f"[{status}] {r.id}: {r.question}")

correct = sum(1 for r in summary.results if r.correct)
executed = sum(1 for r in summary.results if r.executed)
print()
print(f"Execution accuracy:      {summary.execution_accuracy:.1%} ({correct}/{summary.total})")
print(f"Execution success rate:  {summary.execution_success_rate:.1%} ({executed}/{summary.total})")

[PASS] q001: How many customers are there?
[PASS] q002: How many products are there?
[PASS] q003: How many customers are from the USA?
[PASS] q004: What are the top 3 products by total quantity sold?
[PASS] q005: Which customer has spent the most money overall?
[PASS] q006: What is the average order value?
[PASS] q007: How many orders have a status of completed?
[PASS] q008: How many products are in each category?
[PASS] q009: What is the total revenue from all orders?
[PASS] q010: List all distinct countries customers are from.
[PASS] q011: How many orders were placed in 2023?
[PASS] q012: What is the cheapest product?
[PASS] q013: What is the most expensive product?
[PASS] q014: What is Alice's email address?
[PASS] q015: How many products are in the Electronics category?
[PASS] q016: Which product category generated the most total revenue?

Execution accuracy:      100.0% (16/16)
Execution success rate:  100.0% (16/16)


## 16. Bring your own database — the "universal" claim, proven

Every section so far has run against the bundled demo schema
(`customers`/`products`/`orders`/`order_items`). The point of this whole
project is that none of layers 3–4 (knowledge graph, glossary, semantic
retrieval, grounding, the agent graph itself) know anything about *that*
specific schema — they're derived from whatever database is connected, at
bootstrap time. This section proves it by pointing `bootstrap()` at a
brand-new, two-table library/borrower schema this code has never seen
before, with zero extra configuration.

(A real file-based SQLite database is used here rather than
`sqlite:///:memory:`, because two separate `DatabaseConnector` instances
against `:memory:` each get their own private, unshared database — a file
path is what lets the seeding connector and `bootstrap()`'s own connector
see the same data.)

In [33]:
library_db_path = os.path.join(_work_dir, "library.db")
library_url = f"sqlite:///{library_db_path}"

seed_connector = DatabaseConnector(library_url, read_only=False)
seed_connector.execute_ddl(
    """
    CREATE TABLE books (
        book_id INTEGER PRIMARY KEY,
        title   TEXT NOT NULL,
        author  TEXT NOT NULL,
        genre   TEXT NOT NULL
    )
    """
)
seed_connector.execute_ddl(
    """
    CREATE TABLE borrowers (
        borrower_id   INTEGER PRIMARY KEY,
        book_id       INTEGER NOT NULL REFERENCES books(book_id),
        borrower_name TEXT NOT NULL,
        borrowed_on   DATE NOT NULL
    )
    """
)
seed_connector.execute_ddl(
    """
    INSERT INTO books VALUES
      (1, 'Dune', 'Frank Herbert', 'Sci-Fi'),
      (2, 'Foundation', 'Isaac Asimov', 'Sci-Fi'),
      (3, 'The Hobbit', 'J.R.R. Tolkien', 'Fantasy')
    """
)
seed_connector.execute_ddl(
    """
    INSERT INTO borrowers VALUES
      (1, 1, 'Mina', '2024-01-05'),
      (2, 3, 'Owen', '2024-02-11'),
      (3, 1, 'Priya', '2024-03-02')
    """
)

library_llm = llm_for(
    "SIMPLE",
    "SELECT genre, COUNT(*) AS n FROM books GROUP BY genre",
    "VALID",
    "There are 2 Sci-Fi books and 1 Fantasy book.",
)

library_ctx = bootstrap(database_url=library_url, llm=library_llm, seed_demo=False, enable_metadata_enrichment=False)

print("Auto-discovered tables:", list(library_ctx.schema.tables.keys()))
print()
print(library_ctx.describe_knowledge_graph())

library_result = library_ctx.ask("How many books are there in each genre?")
print("\nGenerated SQL:\n", library_result["generated_sql"])
library_result["execution_result"]

Auto-discovered tables: ['books', 'borrowers']

## Table Relationships (Knowledge Graph)
  books.book_id = borrowers.book_id  (declared FK)



Generated SQL:
 SELECT genre, COUNT(*) AS n FROM books GROUP BY genre


,genre,n
0,Fantasy,1
1,Sci-Fi,2


## 17. Try it yourself, and switching to offline/mock mode

Two things worth doing from here:

- **Ask your own question** — the cell below is a self-contained
  `bootstrap()` + `ctx.ask(...)` call, exactly like §3's quickstart, using
  whatever `USE_REAL_LLM` mode is currently active (check the Setup
  cell's printed value at the very top of the notebook). Change
  `try_your_own_question` and re-run the cell.
- **Switch to offline/zero-key mode** — set `USE_REAL_LLM = False` in the
  Setup cell (§1.1) and re-run the notebook top to bottom. Every
  `llm_for(...)` / `llm_for_diverse(...)` call throughout switches to the
  scripted, deterministic stand-in described in §1 — no other code changes
  needed anywhere in this notebook, and no API key or network access
  required.

In [34]:
if USE_REAL_LLM and not (os.getenv("GROQ_API_KEY") or os.getenv("LLM_PROVIDER")):
    print("USE_REAL_LLM is True but no provider looks configured (no GROQ_API_KEY /")
    print("LLM_PROVIDER) -- this cell will likely fail. Either configure a provider")
    print("(cp .env.example .env) or set USE_REAL_LLM = False in the Setup cell above.")
    print()

try_your_own_question = "What are the top 3 products by total revenue?"

try_llm = llm_for(
    "MODERATE",
    (
        "SELECT p.name, SUM(oi.quantity * oi.unit_price) AS revenue "
        "FROM order_items oi JOIN products p ON p.product_id = oi.product_id "
        "GROUP BY p.product_id ORDER BY revenue DESC LIMIT 3"
    ),
    "VALID",
    "The top 3 products by total revenue are shown below.",
)
try_ctx = bootstrap(seed_demo=True, llm=try_llm, enable_metadata_enrichment=False)

try_result = try_ctx.ask(try_your_own_question)
print("SQL:\n", try_result["generated_sql"])
print("\nAnswer:", try_result["final_answer"])
try_result["execution_result"]

SQL:
 SELECT p.name, SUM(oi.quantity * oi.unit_price) AS revenue FROM order_items oi JOIN products p ON p.product_id = oi.product_id GROUP BY p.product_id ORDER BY revenue DESC LIMIT 3

Answer: The top 3 products by total revenue are shown below.


,name,revenue
0,Laptop Pro,5199.96
1,Standing Desk,999.98
2,Desk Chair,599.98


## 18. Summary and where to go next

```mermaid
flowchart TD
    QIN(["question in"]) --> SEL[select_schema]
    SEL -->|"TF-IDF ranking + KG joins<br/>+ glossary + grounding hints<br/>+ conversation context"| CLS[classify_complexity]
    CLS -->|"COMPLEX + decomposition enabled"| DECOMP[decompose_sql]
    CLS -->|"otherwise"| GEN[generate_sql]
    DECOMP --> VOTE[select_best_candidate]
    GEN -->|"1 candidate if SIMPLE,<br/>N candidates otherwise"| VOTE
    VOTE -->|"executes every candidate,<br/>majority-result wins"| EXEC["execute_sql<br/>read-only enforced by default"]
    EXEC --> VAL[validate_result]
    VAL --> FMT[format_answer]
    FMT --> MEM[store_memory]
    MEM --> QOUT(["answer out"])
```

What this notebook walked through, in order: the universal connector and
schema discovery (§4) → the knowledge graph, business glossary, semantic
retrieval, and value grounding that make schema linking work on an unseen
database (§5–§8) → every agent node called individually (§9) → the
compiled graph with self-consistency voting (§10), query decomposition
(§11), SQL safety guardrails (§12), multi-turn conversation (§13), and
pluggable providers (§14) → the evaluation harness that actually measures
whether it's working (§15) → proof the whole thing generalizes to a
database it has never seen (§16).

**Where to go next:**

- [`README.md`](../README.md) — the same architecture, as prose, tables,
  and rendered diagrams, plus the full environment-variable reference.
- [`CONTRIBUTING.md`](../CONTRIBUTING.md) — development setup, the optional
  extras (`safety`, `fuzzy`, `openai`, `anthropic`, `ollama`), and how to
  run the test suite and evaluation harness.
- `tests/` — the same mocked-LLM pattern used throughout this notebook,
  applied as unit tests for every module touched here.
- `main.py` / `app.py` — the CLI and Streamlit UI built on top of exactly
  the `bootstrap()` / `AgentContext.ask()` calls this notebook used.